# Premier League Match Outcome Predictor — Step 1 & 2: Collect + Clean

| | |
|---|---|
| **Purpose** | Merge the per-season football-data.co.uk files into one chronological, sanity-checked match table with a stable odds schema. |
| **Input** | `data/<season>.csv` for 2000-01 … 2026-27 (raw football-data.co.uk exports). |
| **Output** | `data/matches_clean.parquet` (git-ignored; regenerate by running this notebook). |
| **Next** | `features.ipynb` |

Predict match outcomes (H / D / A) from historical data. Priority: correct and defensible, no future leakage.

**This notebook** merges the season CSVs into one chronological, sanity-checked table and
writes it to `data/matches_clean.parquet`.

Feature engineering (Step 3) lives in **`features.ipynb`**, which loads that parquet.

The one rule that governs the whole project: **a feature must be computable before the
match is played.** Never use a match's own stats as an input to its own row.


In [ ]:
import os

import numpy as np
import pandas as pd

# The loaders live in pl_features.py so that this notebook, features.ipynb and predict_upcoming.py
# share one implementation: read_season_csv() truncates the ragged old files to their header width,
# harmonise_odds() collapses the era-specific bookmaker columns into one stable schema
#   mkt_{H,D,A}       pre-match 1X2 consensus   (Avg* 2019+ -> BbAv* 2005-18 -> B365* 2002+)
#   mkt_{over25,under25}  Over/Under 2.5 consensus
#   close_{H,D,A}     closing 1X2 (Pinnacle preferred) -> the sharpest signal
from pl_features import KEEP_COLS, ODDS_GROUPS, harmonise_odds, load_matches, read_season_csv, season_files

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA_DIR = "data"
print("loaders imported from pl_features.py; canonical odds schema:",
      "mkt_{H,D,A}, mkt_{over25,under25}, close_{H,D,A}")


## Step 1 — Collect

The raw files have been renamed to `data/<season>.csv` (`2000-01.csv` … `2026-27.csv`).
Seasons before 2000-01 had no match stats (shots, corners, cards) and were deleted —
a stats-based model can't use them.

`2026-27.csv` is the current season, only ~20 matches played so far — kept as a genuine
out-of-sample set for later.


In [ ]:
# only the season files (data/ also holds team_season_strength.csv etc.)
files = season_files(DATA_DIR)

catalog = []
for path in files:
    raw = read_season_csv(path).dropna(subset=["Date"])
    dates = pd.to_datetime(raw["Date"], dayfirst=True, format="mixed")
    odds = harmonise_odds(raw)
    catalog.append({
        "season": os.path.splitext(os.path.basename(path))[0],
        "first": dates.min().date(),
        "last": dates.max().date(),
        "n_matches": len(raw),
        "has_stats": raw["HS"].notna().any() if "HS" in raw.columns else False,
        "mkt_1x2": f"{odds.mkt_H.notna().mean():.0%}",
        "mkt_ou25": f"{odds.mkt_over25.notna().mean():.0%}",
        "closing": f"{odds.close_H.notna().mean():.0%}",
        "xg": raw["HxG"].notna().any() if "HxG" in raw.columns else False,
    })

catalog = pd.DataFrame(catalog)
print(f"{len(catalog)} seasons, {catalog.n_matches.sum()} matches total")
catalog


## Step 2 — Clean & merge

Per season file:
- keep only `KEEP_COLS` (odds columns missing in the earliest seasons — tolerated),
- parse `Date` → datetime, tag `Season`,
- coerce numeric columns,
- drop matches with no result (`FTR`).

Then concatenate, sort by `Date`, and reset the index. `Season` is kept so the
train/test split can be by season later (never a random shuffle — that leaks the future).


In [ ]:
# load_season() keeps KEEP_COLS (+ kickoff Time where the file has it), parses UK dates, tags the
# season, coerces numerics, adds the canonical odds columns, and drops rows without a result.
matches = load_matches(DATA_DIR)

print(matches.shape)
print("odds coverage —",
      f"mkt_1x2 {matches.mkt_H.notna().mean():.1%},",
      f"O/U 2.5 {matches.mkt_over25.notna().mean():.1%},",
      f"closing {matches.close_H.notna().mean():.1%}")
matches.head()


## Sanity checks

Before building any features, confirm the merged table is sound:
- 380 matches per completed season, ~20 for 2026-27
- every team plays 38 games (19 home + 19 away) per season
- `FTR` is consistent with `FTHG` vs `FTAG`
- outcome distribution — roughly 45% H / 25% D / 30% A is the known baseline
- where the missing values are (older seasons lack odds; a few stat cells missing)


In [ ]:
# matches per season
print("matches per season:")
print(matches.groupby("Season").size().to_string())

# FTR vs goals consistency
derived = pd.Series("D", index=matches.index)
derived[matches.FTHG > matches.FTAG] = "H"
derived[matches.FTHG < matches.FTAG] = "A"
mismatch = (derived != matches.FTR).sum()
print(f"\nFTR inconsistent with goals: {mismatch}")

# outcome distribution
print("\noutcome distribution (all seasons):")
print((matches.FTR.value_counts(normalize=True) * 100).round(1).to_string())


In [ ]:
# every team should play 38 games per completed season
games_played = (
    pd.concat([
        matches[["Season", "HomeTeam"]].rename(columns={"HomeTeam": "Team"}),
        matches[["Season", "AwayTeam"]].rename(columns={"AwayTeam": "Team"}),
    ])
    .groupby(["Season", "Team"]).size()
)
completed = games_played.index.get_level_values("Season") != "2026-27"
bad = games_played[completed & (games_played != 38)]
print(f"team-seasons without exactly 38 games (excl. 2026-27): {len(bad)}")
if len(bad):
    print(bad.to_string())

# missing values by column
print("\nmissing values by column:")
print(matches.isna().sum()[lambda s: s > 0].to_string())


## Save the clean table

Write `matches` to Parquet. `features.ipynb` loads this — the two notebooks stay
decoupled, and re-running feature work doesn't re-parse 27 CSVs each time.


In [ ]:
CLEAN_PATH = os.path.join(DATA_DIR, "matches_clean.parquet")
matches.to_parquet(CLEAN_PATH, index=False)
print(f"wrote {CLEAN_PATH}  ({matches.shape[0]} rows, {matches.shape[1]} cols)")
